
# FlotIQ | Machine Learning

## 03. Predicción del precio de la electricidad

### Objetivo
Entrenar y comparar diferentes modelos de Machine Learning
para predecir el precio mayorista de OMIE con hasta
48 horas de antelación.

### Modelos
- Ridge Regression
- Random Forest
- HistGradientBoosting
- XGBoost

### Evaluación
- Validación temporal
- MAE y RMSE
- Comparación con el baseline semanal
- Análisis de errores en horas caras y baratas

**Mejor baseline actual: 27,66 €/MWh de MAE.**


In [1]:

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

# Localizar la raíz del proyecto
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "omie_ml_hourly.parquet"
)

df = pd.read_parquet(DATA_PATH)

df["datetime_utc"] = pd.to_datetime(
    df["datetime_utc"],
    utc=True
)

df = df.sort_values("datetime_utc").reset_index(drop=True)

print("Dataset cargado correctamente")
print("Filas:", len(df))
print("Columnas:", len(df.columns))


Dataset cargado correctamente
Filas: 32640
Columnas: 15


In [2]:
# VARIABLES PREDICTORAS
# ==========================================

FEATURES = [
    "hour_sin",
    "hour_cos",
    "month_sin",
    "month_cos",
    "day_of_week",
    "is_weekend",
    "price_lag_72h",
    "price_lag_168h",
]

TARGET = "price_eur_mwh"

# Reservar últimos 90 días como test final
test_start = (
    df["datetime_utc"].max()
    - pd.Timedelta(days=90)
    + pd.Timedelta(hours=1)
)

train_val = df[
    df["datetime_utc"] < test_start
].copy()

test = df[
    df["datetime_utc"] >= test_start
].copy()

# Últimos 60 días del periodo anterior
# como validación
val_start = (
    train_val["datetime_utc"].max()
    - pd.Timedelta(days=60)
    + pd.Timedelta(hours=1)
)

train = train_val[
    train_val["datetime_utc"] < val_start
].copy()

val = train_val[
    train_val["datetime_utc"] >= val_start
].copy()

X_train = train[FEATURES]
y_train = train[TARGET]

X_val = val[FEATURES]
y_val = val[TARGET]

print("Entrenamiento:", len(train), "filas")
print("Validación:", len(val), "filas")
print("Test reservado:", len(test), "filas")

assert train["datetime_utc"].max() < val["datetime_utc"].min()
assert val["datetime_utc"].max() < test["datetime_utc"].min()

print("\nDivisión temporal correcta")


Entrenamiento: 29040 filas
Validación: 1440 filas
Test reservado: 2160 filas

División temporal correcta


In [3]:
# FLOTIQ | RIDGE REGRESSION
# ==========================================

alphas = [0.1, 1, 10, 100, 1000]

resultados = []
modelos = {}

for alpha in alphas:

    modelo = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=alpha))
    ])

    modelo.fit(X_train, y_train)

    predicciones = modelo.predict(X_val)

    mae = mean_absolute_error(
        y_val, predicciones
    )

    rmse = np.sqrt(
        mean_squared_error(y_val, predicciones)
    )

    resultados.append({
        "alpha": alpha,
        "MAE": round(mae, 2),
        "RMSE": round(rmse, 2)
    })

    modelos[alpha] = modelo

resultados_ridge = pd.DataFrame(resultados)

display(
    resultados_ridge.sort_values("MAE")
)

# Mejor configuración según validación
mejor_alpha = resultados_ridge.loc[
    resultados_ridge["MAE"].idxmin(),
    "alpha"
]

mejor_ridge = modelos[mejor_alpha]

# Baseline semanal en el mismo periodo
mae_baseline_val = mean_absolute_error(
    y_val,
    val["price_lag_168h"]
)

print(f"\nMejor alpha: {mejor_alpha}")
print(f"MAE baseline semanal (validación): {mae_baseline_val:.2f} €/MWh")
print(
    "MAE mejor Ridge (validación): "
    f"{resultados_ridge['MAE'].min():.2f} €/MWh"
)


,alpha,MAE,RMSE
0,0.1,21.35,26.46
1,1.0,21.36,26.46
2,10.0,21.36,26.46
3,100.0,21.38,26.47
4,1000.0,21.59,26.56



Mejor alpha: 0.1
MAE baseline semanal (validación): 21.37 €/MWh
MAE mejor Ridge (validación): 21.35 €/MWh


In [4]:
# FLOTIQ | RANDOM FOREST
# ==========================================

from sklearn.ensemble import RandomForestRegressor

configuraciones = [
    {"max_depth": 8, "min_samples_leaf": 10},
    {"max_depth": 14, "min_samples_leaf": 5},
]

resultados_rf = []
modelos_rf = {}

for config in configuraciones:

    modelo = RandomForestRegressor(
        n_estimators=150,
        max_depth=config["max_depth"],
        min_samples_leaf=config["min_samples_leaf"],
        random_state=42,
        n_jobs=-1,
    )

    modelo.fit(X_train, y_train)

    predicciones = modelo.predict(X_val)

    mae = mean_absolute_error(y_val, predicciones)

    rmse = np.sqrt(
        mean_squared_error(y_val, predicciones)
    )

    resultados_rf.append({
        "max_depth": config["max_depth"],
        "min_samples_leaf": config["min_samples_leaf"],
        "MAE": round(mae, 2),
        "RMSE": round(rmse, 2),
    })

    modelos_rf[config["max_depth"]] = modelo

resultados_rf_df = pd.DataFrame(resultados_rf)

display(resultados_rf_df.sort_values("MAE"))

# Seleccionar mejor configuración
mejor_fila = resultados_rf_df.loc[
    resultados_rf_df["MAE"].idxmin()
]

mejor_rf = modelos_rf[int(mejor_fila["max_depth"])]

print("\nCOMPARACIÓN DE MODELOS")
print("-" * 40)
print(f"Baseline semanal: {mae_baseline_val:.2f} €/MWh")
print(f"Ridge Regression: {resultados_ridge['MAE'].min():.2f} €/MWh")
print(f"Random Forest:    {mejor_fila['MAE']:.2f} €/MWh")


,max_depth,min_samples_leaf,MAE,RMSE
0,8,10,20.33,26.57
1,14,5,20.58,27.27



COMPARACIÓN DE MODELOS
----------------------------------------
Baseline semanal: 21.37 €/MWh
Ridge Regression: 21.35 €/MWh
Random Forest:    20.33 €/MWh


In [5]:
# FLOTIQ | HIST GRADIENT BOOSTING
# ==========================================

from sklearn.ensemble import HistGradientBoostingRegressor

configuraciones_hgb = [
    {"learning_rate": 0.05, "max_iter": 200},
    {"learning_rate": 0.10, "max_iter": 200},
    {"learning_rate": 0.05, "max_iter": 400},
]

resultados_hgb = []
modelos_hgb = {}

for i, config in enumerate(configuraciones_hgb):

    modelo = HistGradientBoostingRegressor(
        learning_rate=config["learning_rate"],
        max_iter=config["max_iter"],
        max_leaf_nodes=15,
        min_samples_leaf=30,
        l2_regularization=1.0,
        early_stopping=False,
        random_state=42,
    )

    modelo.fit(X_train, y_train)

    predicciones = modelo.predict(X_val)

    mae = mean_absolute_error(
        y_val, predicciones
    )

    rmse = np.sqrt(
        mean_squared_error(y_val, predicciones)
    )

    resultados_hgb.append({
        "learning_rate": config["learning_rate"],
        "max_iter": config["max_iter"],
        "MAE": round(mae, 2),
        "RMSE": round(rmse, 2),
    })

    modelos_hgb[i] = modelo

resultados_hgb_df = pd.DataFrame(resultados_hgb)

display(
    resultados_hgb_df.sort_values("MAE")
)

mejor_indice = resultados_hgb_df["MAE"].idxmin()
mejor_hgb = modelos_hgb[mejor_indice]

print("\nCOMPARACIÓN DE MODELOS")
print("-" * 45)
print(f"Baseline:      {mae_baseline_val:.2f} €/MWh")
print(f"Ridge:         {resultados_ridge['MAE'].min():.2f} €/MWh")
print(f"Random Forest: {resultados_rf_df['MAE'].min():.2f} €/MWh")
print(f"HistBoosting:  {resultados_hgb_df['MAE'].min():.2f} €/MWh")


,learning_rate,max_iter,MAE,RMSE
0,0.05,200,19.82,26.11
2,0.05,400,20.15,26.73
1,0.10,200,20.29,27.02



COMPARACIÓN DE MODELOS
---------------------------------------------
Baseline:      21.37 €/MWh
Ridge:         21.35 €/MWh
Random Forest: 20.33 €/MWh
HistBoosting:  19.82 €/MWh


In [6]:
# FLOTIQ | XGBOOST
# ==========================================

from xgboost import XGBRegressor

configuraciones_xgb = [
    {
        "max_depth": 3,
        "learning_rate": 0.05,
        "n_estimators": 300
    },
    {
        "max_depth": 5,
        "learning_rate": 0.05,
        "n_estimators": 300
    },
    {
        "max_depth": 3,
        "learning_rate": 0.03,
        "n_estimators": 500
    },
]

resultados_xgb = []
modelos_xgb = {}

for i, config in enumerate(configuraciones_xgb):

    modelo = XGBRegressor(
        max_depth=config["max_depth"],
        learning_rate=config["learning_rate"],
        n_estimators=config["n_estimators"],
        objective="reg:squarederror",
        subsample=0.8,
        colsample_bytree=0.9,
        reg_lambda=5.0,
        tree_method="hist",
        random_state=42,
        n_jobs=4
    )

    modelo.fit(X_train, y_train)

    predicciones = modelo.predict(X_val)

    mae = mean_absolute_error(
        y_val,
        predicciones
    )

    rmse = np.sqrt(
        mean_squared_error(y_val, predicciones)
    )

    resultados_xgb.append({
        "max_depth": config["max_depth"],
        "learning_rate": config["learning_rate"],
        "n_estimators": config["n_estimators"],
        "MAE": round(mae, 2),
        "RMSE": round(rmse, 2)
    })

    modelos_xgb[i] = modelo

resultados_xgb_df = pd.DataFrame(resultados_xgb)

display(
    resultados_xgb_df.sort_values("MAE")
)

# Mejor configuración
mejor_indice_xgb = resultados_xgb_df["MAE"].idxmin()

mejor_xgb = modelos_xgb[mejor_indice_xgb]

# Comparación global
comparacion = pd.DataFrame([
    {"Modelo": "Baseline semanal", "MAE": mae_baseline_val},
    {"Modelo": "Ridge", "MAE": resultados_ridge["MAE"].min()},
    {"Modelo": "Random Forest", "MAE": resultados_rf_df["MAE"].min()},
    {"Modelo": "HistGradientBoosting", "MAE": resultados_hgb_df["MAE"].min()},
    {"Modelo": "XGBoost", "MAE": resultados_xgb_df["MAE"].min()}
])

print("\nRANKING FLOTIQ")
display(
    comparacion.sort_values("MAE").round(2)
)


,max_depth,learning_rate,n_estimators,MAE,RMSE
2,3,0.03,500,20.25,26.01
0,3,0.05,300,20.26,25.95
1,5,0.05,300,20.54,26.99



RANKING FLOTIQ


,Modelo,MAE
3,HistGradientBoosting,19.82
4,XGBoost,20.25
2,Random Forest,20.33
1,Ridge,21.35
0,Baseline semanal,21.37


In [7]:
# FLOTIQ | VALIDACIÓN TEMPORAL
# ============================================

from sklearn.base import clone

# Utilizamos las configuraciones seleccionadas
# anteriormente y mantenemos el test intacto.
candidatos = {
    "HistGradientBoosting": mejor_hgb,
    "XGBoost": mejor_xgb,
}

resultados_cv = []

DIAS_VALIDACION = 60
NUM_FOLDS = 3

for fold in range(NUM_FOLDS):

    # Ventanas cronológicas anteriores al test
    dias_restantes = (
        NUM_FOLDS - fold
    ) * DIAS_VALIDACION

    inicio = (
        test_start
        - pd.Timedelta(days=dias_restantes)
    )

    fin = inicio + pd.Timedelta(
        days=DIAS_VALIDACION
    )

    datos_train = df[
        df["datetime_utc"] < inicio
    ].copy()

    datos_val = df[
        (df["datetime_utc"] >= inicio)
        & (df["datetime_utc"] < fin)
    ].copy()

    assert len(datos_val) == 60 * 24

    # Modelo de referencia
    baseline_mae = mean_absolute_error(
        datos_val[TARGET],
        datos_val["price_lag_168h"]
    )

    resultados_cv.append({
        "Fold": fold + 1,
        "Modelo": "Baseline semanal",
        "MAE": baseline_mae,
    })

    # Modelos ML
    for nombre, modelo_base in candidatos.items():

        modelo = clone(modelo_base)

        modelo.fit(
            datos_train[FEATURES],
            datos_train[TARGET]
        )

        predicciones = modelo.predict(
            datos_val[FEATURES]
        )

        mae = mean_absolute_error(
            datos_val[TARGET],
            predicciones
        )

        resultados_cv.append({
            "Fold": fold + 1,
            "Modelo": nombre,
            "MAE": mae,
        })

resultados_cv_df = pd.DataFrame(resultados_cv)

print("RESULTADOS POR VENTANA")
display(
    resultados_cv_df.pivot(
        index="Fold",
        columns="Modelo",
        values="MAE"
    ).round(2)
)

print("\nMAE MEDIO DE LAS 3 VENTANAS")

resumen_cv = (
    resultados_cv_df
    .groupby("Modelo")["MAE"]
    .agg(["mean", "std"])
    .sort_values("mean")
    .round(2)
)

display(resumen_cv)


RESULTADOS POR VENTANA


Modelo,Baseline semanal,HistGradientBoosting,XGBoost
Fold,,,
1,26.89,25.79,26.31
2,31.01,23.48,23.82
3,21.37,19.82,20.25



MAE MEDIO DE LAS 3 VENTANAS


,mean,std
Modelo,,
HistGradientBoosting,23.03,3.01
XGBoost,23.46,3.04
Baseline semanal,26.43,4.84


In [8]:
# FLOTIQ | EVALUACIÓN FINAL EN TEST
# ============================================

from sklearn.base import clone

# Entrenamos con todos los datos anteriores
# al conjunto de prueba.
X_train_final = train_val[FEATURES]
y_train_final = train_val[TARGET]

X_test = test[FEATURES]
y_test = test[TARGET]

# Candidatos seleccionados durante validación
modelos_finales = {
    "HistGradientBoosting": clone(mejor_hgb),
    "XGBoost": clone(mejor_xgb),
}

predicciones_test = test[
    ["datetime_utc", TARGET]
].copy()

# Baseline semanal
predicciones_test["Baseline semanal"] = (
    test["price_lag_168h"].to_numpy()
)

resultados_test = []

# Evaluar baseline
mae = mean_absolute_error(
    y_test,
    predicciones_test["Baseline semanal"]
)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        predicciones_test["Baseline semanal"]
    )
)

resultados_test.append({
    "Modelo": "Baseline semanal",
    "MAE": mae,
    "RMSE": rmse,
})

# Entrenar y evaluar los modelos ML
for nombre, modelo in modelos_finales.items():

    modelo.fit(
        X_train_final,
        y_train_final
    )

    pred = modelo.predict(X_test)

    predicciones_test[nombre] = pred

    mae = mean_absolute_error(
        y_test, pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test, pred
        )
    )

    resultados_test.append({
        "Modelo": nombre,
        "MAE": mae,
        "RMSE": rmse,
    })

resultados_test_df = (
    pd.DataFrame(resultados_test)
    .sort_values("MAE")
    .reset_index(drop=True)
)

print("FLOTIQ | RESULTADOS FINALES EN TEST")
print("-" * 45)

display(
    resultados_test_df.round(2)
)

# Guardar predicciones para análisis posterior
OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "omie_ml_test_predictions.parquet"
)

predicciones_test.to_parquet(
    OUTPUT_PATH,
    index=False
)

print("\nPredicciones guardadas correctamente.")
print("Periodo:", test["datetime_utc"].min(),
      "a", test["datetime_utc"].max())


FLOTIQ | RESULTADOS FINALES EN TEST
---------------------------------------------


,Modelo,MAE,RMSE
0,Baseline semanal,27.66,39.18
1,HistGradientBoosting,38.68,45.90
2,XGBoost,42.00,49.70



Predicciones guardadas correctamente.
Periodo: 2026-07-02 22:00:00+00:00 a 2026-09-30 21:00:00+00:00
